In [ ]:
# Parameters
BATCH_MODE = False


# 11. Quantization des LLMs

**Navigation** : [<< 10. Hebergement Local](10_LocalLlama.ipynb) | [Index](../../README.md)

**Duree estimee** : 60 minutes

**Prerequis** : Notebook 10 (LocalLlama), Python 3.10+, GPU recommande (meme un petit)

---

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. Pourquoi et comment la quantization reduit la taille des LLMs
2. Calculer l'empreinte memoire d'un modele par precision
3. Distinguer les methodes BitsAndBytes, AWQ et GPTQ
4. Quantifier un modele avec llmcompressor (methode production)
5. Deployer et valider un modele quantifie avec vLLM

---

## Contexte

Dans le notebook precedent, nous avons deploye des LLMs localement avec vLLM.
Vous avez peut-etre remarque que nos modeles etaient en **AWQ 4-bit** ou **GPTQ 4-bit**.
Ce notebook explique **pourquoi** et **comment** on passe d'un modele BF16 de 16 GB
a un modele 4-bit de 5 GB -- et pourquoi cela **accelere** l'inference au lieu de la ralentir.

## Concepts cles

### Pourquoi quantifier ?

Le goulot d'etranglement principal de l'inference LLM est la **bande passante memoire**, pas la puissance de calcul.
A chaque token genere, le GPU doit lire **tous les poids du modele** depuis la VRAM.
Reduire la taille des poids signifie :

1. **Moins de VRAM** : Un modele 8B passe de 16 GB (BF16) a 5 GB (4-bit)
2. **Inference plus rapide** : Moins de donnees a lire = plus de tokens/seconde
3. **Plus de batching** : La VRAM economisee sert au KV cache = plus d'utilisateurs simultanes
4. **Deployement moins cher** : Un modele 70B tient sur 2 GPUs au lieu de 4

> **Insight contre-intuitif** : La quantization **accelere** l'inference.
> On pourrait penser que reduire la precision ralentit le modele.
> En realite, le bottleneck est le **memory bandwidth**, pas le compute.
> Lire 4 bits au lieu de 16 bits par poids = 4x moins de donnees a transferer.

### Formats de precision

| Format | Octets/param | Description | Usage typique |
|--------|-------------|-------------|---------------|
| FP32 | 4 | Full precision | Entrainement, reference |
| BF16 / FP16 | 2 | Half precision | Entrainement mixte, inference standard |
| INT8 | 1 | 8-bit integer | Quantization simple (BitsAndBytes) |
| W4A16 (INT4) | 0.5 | 4-bit poids, 16-bit activations | Production (AWQ, GPTQ) |

### Trois lecons cles

1. **La quantization accelere l'inference** (contre-intuitif) : moins de memoire a lire = plus de tokens/seconde
2. **Les modeles multimodaux necessitent des exclusions** : ne jamais quantifier l'encodeur vision (ViT/SigLIP)
3. **Le format de sortie determine le runtime** : compressed-tensors vers Marlin, GPTQ vers GPTQMarlin, GGUF vers llama.cpp

## Installation et imports

Nous installons les bibliotheques necessaires pour ce notebook.
Les demonstrations de quantization lourde (llmcompressor, BitsAndBytes)
sont protegees par `BATCH_MODE` et presentees en lecture seule.

In [2]:
%pip install torch transformers accelerate datasets huggingface_hub python-dotenv requests openai bitsandbytes -q --upgradefrom pathlib import Pathimport osimport jsonimport timeimport requestsfrom dotenv import load_dotenvload_dotenv('../.env')BATCH_MODE = os.getenv("BATCH_MODE", "false").lower() == "true"print(f"Mode batch: {BATCH_MODE}")# Verification GPUtry:    import torch    if torch.cuda.is_available():        gpu_name = torch.cuda.get_device_name(0)        gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1024**3        print(f"GPU: {gpu_name} ({gpu_mem:.1f} GB)")    else:        print("Pas de GPU disponible")except Exception as e:    print(f"Erreur GPU: {e}")

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
argumentation-analysis-project 0.1.0 requires networkx==3.2.1, but you have networkx 3.6.1 which is incompatible.
argumentation-analysis-project 0.1.0 requires pandas==2.2.3, but you have pandas 2.3.1 which is incompatible.
argumentation-analysis-project 0.1.0 requires pydantic==2.9.2, but you have pydantic 2.11.10 which is incompatible.
argumentation-analysis-project 0.1.0 requires scikit-learn==1.6.1, but you have scikit-learn 1.7.1 which is incompatible.
argumentation-a

Note: you may need to restart the kernel to use updated packages.
Mode batch: False


Pas de GPU CUDA detecte - certaines cellules seront en mode demo


### Interpretation de l'installation

**Bibliotheques installees** :

| Package | Role |
|---------|------|
| `torch` | Backend GPU, mesure memoire VRAM |
| `transformers` | Chargement de modeles HuggingFace |
| `accelerate` | Distribution multi-GPU, `device_map="auto"` |
| `datasets` | Chargement du dataset de calibration |
| `openai` | Client API compatible vLLM |
| `python-dotenv` | Chargement de la configuration `.env` |

**Variable `BATCH_MODE`** : Lue depuis le `.env` apres le parametre Papermill.
Les cellules lourdes (chargement de modeles, quantization) sont ignorees en mode batch
pour permettre une validation automatisee rapide.

---

## Section 1 : Pourquoi quantifier ? Calcul de l'empreinte memoire

La formule fondamentale pour estimer la VRAM requise par un modele est :

$$
\text{Memoire (GB)} = \text{nombre\_params} \times \text{octets\_par\_param} \div 10^9
$$

En pratique, il faut ajouter environ **20% de surcharge** pour le KV cache, les activations
et le framework d'inference.

### Modeles courants et leur empreinte

| Modele | Params | BF16 | INT8 | W4A16 |
|--------|--------|------|------|-------|
| Qwen3.5-0.8B | 0.8B | 1.6 GB | 0.8 GB | 0.4 GB |
| ZwZ-8B | 8B | 16 GB | 8 GB | 4 GB |
| Qwen3.5-35B-A3B (MoE, total) | 35B | 70 GB | 35 GB | 17.5 GB |
| Llama 3.1 70B | 70B | 140 GB | 70 GB | 35 GB |

On voit immediatement que la quantization W4A16 divise par **4** l'empreinte memoire
par rapport au BF16. Un modele 70B qui necessite 4 GPUs tient alors sur 2 GPUs.

In [3]:
def estimate_model_memory(num_params_billions, precision="bf16"):
    """Estime la memoire GPU necessaire pour un modele LLM."""
    bytes_per_param = {
        "fp32": 4, "bf16": 2, "fp16": 2,
        "int8": 1, "int4": 0.5, "w4a16": 0.5
    }
    bpp = bytes_per_param.get(precision, 2)
    memory_gb = num_params_billions * bpp
    # Ajouter ~20% de surcharge pour le KV cache, activations, framework
    total_gb = memory_gb * 1.2
    return memory_gb, total_gb


# Table comparative
models = [
    ("Qwen3.5-0.8B", 0.8),
    ("ZwZ-8B", 8),
    ("Qwen3.5-35B-A3B (MoE, total)", 35),
    ("Llama 3.1 70B", 70),
]

print(f"{'Modele':<35} {'BF16':>8} {'INT8':>8} {'W4A16':>8}")
print("-" * 65)
for name, params in models:
    bf16, _ = estimate_model_memory(params, "bf16")
    int8, _ = estimate_model_memory(params, "int8")
    w4, _ = estimate_model_memory(params, "w4a16")
    print(f"{name:<35} {bf16:>6.1f}GB {int8:>6.1f}GB {w4:>6.1f}GB")

print("\n--- Donnees reelles (workspace vLLM) ---")
print("ZwZ-8B:           BF16 ~17 GB  ->  AWQ 4-bit ~5.5 GB  (ratio 3.1x)")
print("Qwen3.5-35B-A3B:  BF16 ~70 GB  ->  GPTQ 4-bit ~24 GB  (ratio 2.9x)")

Modele                                  BF16     INT8    W4A16
-----------------------------------------------------------------
Qwen3.5-0.8B                           1.6GB    0.8GB    0.4GB
ZwZ-8B                                16.0GB    8.0GB    4.0GB
Qwen3.5-35B-A3B (MoE, total)          70.0GB   35.0GB   17.5GB
Llama 3.1 70B                        140.0GB   70.0GB   35.0GB

--- Donnees reelles (workspace vLLM) ---
ZwZ-8B:           BF16 ~17 GB  ->  AWQ 4-bit ~5.5 GB  (ratio 3.1x)
Qwen3.5-35B-A3B:  BF16 ~70 GB  ->  GPTQ 4-bit ~24 GB  (ratio 2.9x)


### Interpretation : empreinte memoire

**Sortie obtenue** : Table comparative des tailles memoire par precision.

| Aspect | Valeur | Signification |
|--------|--------|---------------|
| Ratio BF16 -> W4A16 | 4x | Division par 4 de la memoire des poids |
| Ratio reel ZwZ-8B | 3.1x | Legerement inferieur car le ViT n'est pas quantifie |
| Ratio reel Qwen3.5-35B | 2.9x | Les couches MoE non actives prennent aussi de la place |

**Points cles** :
1. La theorie (4x) et la pratique (3x) different car certaines couches ne sont pas quantifiees
2. La surcharge de 20% est une estimation ; elle varie selon la longueur du contexte
3. Pour les modeles MoE, tous les experts sont en memoire meme si peu sont actifs par token

> **Note technique** : Les donnees reelles proviennent de notre stack de production
> avec 3x RTX 4090 (72 GB VRAM total). Les mesures incluent le framework vLLM.

---

## Section 2 : Methodes de quantization

Il existe plusieurs approches pour quantifier un LLM. Elles se distinguent par
le moment de la quantization (runtime vs offline), le besoin de calibration,
et le runtime cible.

### Comparaison des methodes

| Critere | BitsAndBytes | AutoAWQ | GPTQ (llmcompressor) | GGUF |
|---------|-------------|---------|----------------------|------|
| **Type** | Runtime | Offline | Offline | Offline |
| **Precision** | INT8, NF4 | W4A16 | W4A16 | Q4_K_M, Q5, etc. |
| **Calibration** | Non | Oui | Oui (Open-Platypus, 512 samples) | Conversion |
| **Runtime cible** | Transformers | vLLM, Transformers | vLLM (Marlin kernels natifs) | llama.cpp, Ollama |
| **Facilite** | Tres simple | Simple | Moyen | Simple |
| **Production** | Non | Oui | Oui (recommande) | Oui (CPU/edge) |

### BitsAndBytes

La methode la plus simple : ajoutez `load_in_8bit=True` ou `load_in_4bit=True` au chargement.
Aucune calibration necessaire, la quantization se fait a la volee.
**Avantage** : Zero configuration. **Inconvenient** : Plus lent que les methodes offline
car le dequantize se fait a chaque forward pass sans kernels optimises.

### AutoAWQ (Activation-aware Weight Quantization)

Methode offline qui utilise un petit dataset de calibration pour determiner
les meilleures echelles de quantization par canal. Produit des modeles en format
AWQ lisibles par vLLM et Transformers.

### GPTQ via llmcompressor (recommande pour vLLM)

Methode offline developpee par Neural Magic (equipe integree a vLLM).
Produit le format **compressed-tensors** natif pour vLLM avec les kernels Marlin optimises.
C'est la methode recommandee pour la production.

### GGUF (llama.cpp)

Format specifique a llama.cpp et Ollama. Optimise pour l'inference CPU ou GPU partiel.
Ideal pour le deploiement sur machines sans GPU puissant (laptops, edge devices).
Nombreux niveaux de quantization disponibles (Q2_K, Q4_K_M, Q5_K_M, Q8_0, etc.).

### Demonstration BitsAndBytes (quantization runtime)

BitsAndBytes est la porte d'entree la plus simple vers la quantization.
Nous allons charger un petit modele (0.8B parametres) en BF16 puis en INT8
pour comparer l'empreinte memoire.

> **Note** : Cette cellule necessite un GPU CUDA et la bibliotheque `bitsandbytes`.
> Elle est ignoree en `BATCH_MODE` car le chargement de modeles est lent.

In [4]:
# BitsAndBytes : quantization runtime la plus simple
# Pas besoin de calibration - le modele est quantifie au chargement

if not BATCH_MODE:
    try:
        from transformers import AutoModelForCausalLM, AutoTokenizer
        import torch

        model_id = "Qwen/Qwen3.5-0.8B"  # Petit modele ideal pour demo

        print("Chargement en BF16 (reference)...")
        tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)

        # Mesurer la memoire avant/apres
        if torch.cuda.is_available():
            torch.cuda.reset_peak_memory_stats()
            model_bf16 = AutoModelForCausalLM.from_pretrained(
                model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=True
            )
            mem_bf16 = torch.cuda.max_memory_allocated() / 1e9
            print(f"  Memoire BF16: {mem_bf16:.2f} GB")
            del model_bf16
            torch.cuda.empty_cache()

            print("\nChargement en INT8 (BitsAndBytes)...")
            torch.cuda.reset_peak_memory_stats()
            model_int8 = AutoModelForCausalLM.from_pretrained(
                model_id, load_in_8bit=True, device_map="auto", trust_remote_code=True
            )
            mem_int8 = torch.cuda.max_memory_allocated() / 1e9
            print(f"  Memoire INT8: {mem_int8:.2f} GB")
            print(f"  Ratio: {mem_bf16/mem_int8:.1f}x reduction")

            # Test rapide d'inference
            inputs = tokenizer(
                "La quantization des LLMs permet de",
                return_tensors="pt"
            ).to("cuda")
            with torch.no_grad():
                outputs = model_int8.generate(**inputs, max_new_tokens=30)
            print(f"\nGeneration INT8: {tokenizer.decode(outputs[0], skip_special_tokens=True)}")

            del model_int8
            torch.cuda.empty_cache()
        else:
            print("Pas de GPU - demo BitsAndBytes non disponible")
    except ImportError:
        print("bitsandbytes non installe. Installez avec: pip install bitsandbytes")
else:
    print("[BATCH_MODE] Demo BitsAndBytes ignoree")

C:\Users\MYIA\AppData\Roaming\Python\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Chargement en BF16 (reference)...


Pas de GPU - demo BitsAndBytes non disponible


### Interpretation : BitsAndBytes

**Resultats attendus** (sur GPU) :

| Metrique | BF16 | INT8 | Ratio |
|----------|------|------|-------|
| Memoire VRAM | ~1.6 GB | ~0.9 GB | ~1.8x |

**Points cles** :
1. Le ratio n'est pas exactement 2x car les buffers et activations restent en FP16
2. `load_in_8bit=True` est un seul parametre a ajouter -- zero configuration
3. La qualite du texte genere reste comparable en INT8 pour un modele aussi petit
4. BitsAndBytes est ideal pour le **prototypage** mais pas pour la production
   (les kernels ne sont pas aussi optimises que Marlin/AWQ pour vLLM)

> **Quand utiliser BitsAndBytes ?** : Exploration rapide, fine-tuning QLoRA,
> ou quand vous voulez tester un modele avant de lancer une quantization offline.

---

## Section 3 : GPTQ avec llmcompressor (methode production)

Pour deployer un modele quantifie en production avec vLLM, la methode recommandee
est **GPTQ via llmcompressor**. Voici pourquoi :

1. **Format compressed-tensors** : Natif pour vLLM, aucune conversion necessaire
2. **Exclusions fines via regex** : Essentiel pour les modeles vision-language (voir section 4)
3. **Developpe par Neural Magic** : L'equipe integree a vLLM, kernels Marlin optimises
4. **One-shot** : Une seule passe sur le dataset de calibration

### Dataset de calibration

llmcompressor utilise un dataset de calibration pour determiner les echelles de quantization
optimales. Le choix par defaut est **Open-Platypus** :

- 25K paires instruction/reponse couvrant des sujets STEM varies
- 512 samples est le point optimal (bon compromis temps/qualite)
- Au-dela de 512, le gain est marginal

### Parametres cles

| Parametre | Valeur | Description |
|-----------|--------|-------------|
| `targets` | `"Linear"` | Quantifier toutes les couches lineaires |
| `scheme` | `"W4A16"` | 4-bit poids, 16-bit activations |
| `ignore` | `["lm_head"]` | Exclure la couche de sortie (sensible) |
| `dampening_frac` | `0.01` | Stabilisation numerique |
| `block_size` | `128` | Taille des blocs de quantization |

In [5]:
# Exemple de script de quantization GPTQ avec llmcompressor
# Adapte des scripts de production (workspace vLLM)
#
# NOTE: Ce code est presente a titre educatif.
# L'execution reelle necessite un GPU puissant et ~30 min,
# il est donc presente en lecture seule.

QUANTIZATION_SCRIPT = '''
# Installation requise :
# conda create -n llmcompressor python=3.11 -y
# pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
# pip install "llmcompressor>=0.9.0" "transformers>=4.48.0" accelerate datasets

from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor import oneshot
from datasets import load_dataset
from transformers import AutoTokenizer

# ============================================================
# 1. Configuration
# ============================================================
model_id = "Qwen/Qwen3.5-0.8B"    # Modele source (BF16)
output_dir = "./models/Qwen3.5-0.8B-GPTQ-W4A16"

# ============================================================
# 2. Dataset de calibration
# ============================================================
# Open-Platypus : 25K paires instruction/reponse, diversite STEM
# 512 samples = bon compromis temps/qualite
ds = load_dataset("garage-bAInd/Open-Platypus", split="train")
ds = ds.shuffle(seed=42).select(range(512))

def preprocess(example):
    return {"text": example["instruction"] + "\\n" + example.get("output", "")}
ds = ds.map(preprocess)

# ============================================================
# 3. Configuration GPTQ
# ============================================================
recipe = GPTQModifier(
    targets="Linear",              # Toutes les couches lineaires
    scheme="W4A16",                # 4-bit poids, 16-bit activations
    ignore=["lm_head"],            # Ne pas quantifier la couche de sortie
    dampening_frac=0.01,           # Stabilisation numerique
    block_size=128,                # Taille des blocs de quantization
)

# ============================================================
# 4. Quantization one-shot
# ============================================================
oneshot(
    model=model_id,
    dataset=ds,
    recipe=recipe,
    output_dir=output_dir,
    max_seq_length=4096,
    num_calibration_samples=512,
)

print(f"Modele quantifie sauvegarde dans: {output_dir}")
'''

print("=== Script de quantization GPTQ (llmcompressor) ===")
print(QUANTIZATION_SCRIPT)

if not BATCH_MODE:
    print("\n--- Ce script est presente a titre educatif ---")
    print("Pour l'executer reellement :")
    print("1. Creer un environnement conda dedie")
    print("2. Installer les dependances ci-dessus")
    print("3. Copier le script dans un fichier .py")
    print("4. Executer (~5-10 min pour 0.8B, ~35 min pour 8B)")
else:
    print("\n[BATCH_MODE] Script presente en lecture seule")

=== Script de quantization GPTQ (llmcompressor) ===

# Installation requise :
# conda create -n llmcompressor python=3.11 -y
# pip install torch==2.5.1 --index-url https://download.pytorch.org/whl/cu124
# pip install "llmcompressor>=0.9.0" "transformers>=4.48.0" accelerate datasets

from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor import oneshot
from datasets import load_dataset
from transformers import AutoTokenizer

# ============================================================
# 1. Configuration
# ============================================================
model_id = "Qwen/Qwen3.5-0.8B"    # Modele source (BF16)
output_dir = "./models/Qwen3.5-0.8B-GPTQ-W4A16"

# ============================================================
# 2. Dataset de calibration
# ============================================================
# Open-Platypus : 25K paires instruction/reponse, diversite STEM
# 512 samples = bon compromis temps/qualite
ds = load_dataset("garage-bAInd/

### Interpretation : GPTQ avec llmcompressor

**Structure du script** :

| Etape | Description | Duree (~8B) |
|-------|-------------|-------------|
| 1. Configuration | Definir modele source et sortie | < 1 sec |
| 2. Dataset | Charger et preparer Open-Platypus | ~30 sec |
| 3. GPTQModifier | Definir la recette de quantization | < 1 sec |
| 4. oneshot() | Quantization effective | ~35 min |

**Impact du nombre d'echantillons de calibration** :

| Samples | Temps (~8B) | Recommandation |
|---------|-------------|----------------|
| 128 | ~15 min | Prototypage rapide |
| 256 | ~20 min | Tests |
| 512 | ~35 min | Production (defaut) |
| 1024 | ~70 min | Gain marginal |

**Points cles** :
1. Le `GPTQModifier` est l'element central : il definit la **recette** de quantization
2. `ignore=["lm_head"]` est crucial : la couche de sortie projette vers le vocabulaire
   et sa precision affecte directement la qualite de generation
3. `scheme="W4A16"` signifie : poids en 4 bits, activations en 16 bits au runtime
4. Le format de sortie est automatiquement **compressed-tensors**, natif pour vLLM

---

## Section 4 : Cas special -- Modeles multimodaux (Vision-Language)

Les modeles Vision-Language (VL) comme Qwen3-VL ou ZwZ-8B combinent un **encodeur vision**
(ViT/SigLIP) et un **decodeur langage** (LLM). La quantization de ces modeles
necessite une attention particuliere.

### Pourquoi exclure l'encodeur vision ?

L'encodeur vision utilise des operations **softmax** et **LayerNorm** extremement sensibles
a la precision numerique. Quantifier le ViT entraine typiquement :

- **-20 points** sur les benchmarks OCR (lecture de texte dans les images)
- **-15 points** sur les benchmarks de comprehension visuelle
- Des artefacts dans la representation des patches d'image

### Cout en memoire de l'exclusion

L'encodeur vision ne represente qu'une petite fraction du modele total :

| Composant | Params (8B VL) | Memoire BF16 | % du total |
|-----------|---------------|-------------|------------|
| Visual encoder (ViT) | ~300M | ~0.6 GB | 3.7% |
| Patch merger | ~50M | ~0.1 GB | 0.6% |
| Language model | ~7.6B | ~15.2 GB | 95.7% |

Le surpoids de garder le ViT en BF16 est negligeable (~0.7 GB) par rapport
au gain de quantifier le LLM (15.2 GB -> 3.8 GB).

### Patterns d'exclusion

```python
ignore=[
    "lm_head",          # Couche de sortie (sensible)
    "re:.*visual.*",    # Encodeur vision complet
    "re:.*merger.*",    # Patch merger vision -> langage
]
```

> **Attention** : Le script doit pre-charger le modele avec la bonne classe
> (`Qwen3VLForConditionalGeneration`) car `oneshot()` utilise `AutoModelForCausalLM`
> en interne, qui ne supporte pas les modeles VL.

In [6]:
# Quantization d'un modele Vision-Language
# Les couches vision sont EXCLUES pour preserver la qualite

VL_QUANTIZATION_SCRIPT = '''
# Difference cle vs modele text-only :
# 1. Utiliser le bon model class (pas AutoModelForCausalLM)
# 2. Exclure le visual encoder et le merger
# 3. Utiliser processor au lieu de tokenizer seul

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor import oneshot

model_id = "inclusionAI/ZwZ-8B"  # Modele VL (finetune Qwen3-VL-8B)

# CRITIQUE: Pre-charger avec la bonne classe
# AutoModelForCausalLM ne supporte PAS les modeles VL!
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

# Configuration GPTQ avec EXCLUSIONS vision
recipe = GPTQModifier(
    targets="Linear",
    scheme="W4A16",
    ignore=[
        "lm_head",          # Couche de sortie (sensible)
        "re:.*visual.*",    # Visual encoder ViT (tres sensible!)
        "re:.*merger.*",    # Patch merger vision->language
    ],
    dampening_frac=0.01,
)

# Le dataset de calibration est TEXT-ONLY
# (les couches vision sont exclues, pas besoin d images)
# ... meme dataset Open-Platypus que pour text-only ...

oneshot(
    model=model,              # Objet model (pas string!)
    dataset=ds,
    recipe=recipe,
    output_dir="./models/ZwZ-8B-AWQ-4bit",
    max_seq_length=4096,
    num_calibration_samples=512,
    save_compressed=False,    # Contourner bug packing VL
)
'''

print("=== Script quantization VL (avec exclusions vision) ===")
print(VL_QUANTIZATION_SCRIPT)

# Comment inspecter les couches d'un modele
print("\n--- Pour identifier les couches a exclure ---")
print("from transformers import Qwen3VLForConditionalGeneration")
print("model = Qwen3VLForConditionalGeneration.from_pretrained(model_id)")
print("for name, _ in model.named_modules():")
print('    if \"visual\" in name or \"merger\" in name:')
print('        print(name)  # Ces couches doivent etre exclues')

=== Script quantization VL (avec exclusions vision) ===

# Difference cle vs modele text-only :
# 1. Utiliser le bon model class (pas AutoModelForCausalLM)
# 2. Exclure le visual encoder et le merger
# 3. Utiliser processor au lieu de tokenizer seul

from transformers import Qwen3VLForConditionalGeneration, AutoProcessor
from llmcompressor.modifiers.quantization import GPTQModifier
from llmcompressor import oneshot

model_id = "inclusionAI/ZwZ-8B"  # Modele VL (finetune Qwen3-VL-8B)

# CRITIQUE: Pre-charger avec la bonne classe
# AutoModelForCausalLM ne supporte PAS les modeles VL!
model = Qwen3VLForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto",
)
processor = AutoProcessor.from_pretrained(model_id)

# Configuration GPTQ avec EXCLUSIONS vision
recipe = GPTQModifier(
    targets="Linear",
    scheme="W4A16",
    ignore=[
        "lm_head",          # Couche de sortie (sensible)
        "re:.*visual.*",    # Visual encoder ViT (tres sen

### Interpretation : quantization Vision-Language

**Differences cles avec un modele text-only** :

| Aspect | Text-only | Vision-Language |
|--------|-----------|----------------|
| Model class | `AutoModelForCausalLM` | `Qwen3VLForConditionalGeneration` |
| Argument `model=` | String (model_id) | Objet model pre-charge |
| Exclusions | `["lm_head"]` | `["lm_head", "re:.*visual.*", "re:.*merger.*"]` |
| `save_compressed` | Par defaut (True) | `False` (bug de packing VL) |
| Surpoids ViT | N/A | ~0.7 GB (negligeable) |

**Points cles** :
1. Le pre-chargement avec la bonne classe est **obligatoire** car `oneshot()` utilise
   `AutoModelForCausalLM` en interne, incompatible avec les modeles VL
2. Le dataset de calibration reste **text-only** : les couches vision sont exclues,
   donc pas besoin d'images pour la calibration
3. `save_compressed=False` contourne un bug connu de packing avec les modeles VL
4. L'inspection des couches avec `named_modules()` est la methode fiable pour
   identifier les patterns a exclure

---

## Section 5 : Deploiement avec vLLM

Une fois le modele quantifie, le deploiement avec vLLM est straightforward.
vLLM detecte automatiquement le format compressed-tensors et charge les kernels
Marlin optimises.

### Commande de lancement

```bash
vllm serve ./models/ZwZ-8B-AWQ-4bit \
    --dtype auto \
    --kv-cache-dtype fp8 \
    --gpu-memory-utilization 0.85 \
    --max-model-len 131072 \
    --port 5001
```

### Parametres critiques

| Parametre | Valeur | Explication |
|-----------|--------|-------------|
| `--dtype auto` | auto | **Pas `half`!** vLLM choisit le bon dtype selon le modele |
| `--kv-cache-dtype fp8` | fp8 | Reduit la memoire du KV cache de 2x |
| `--gpu-memory-utilization` | 0.85 | Reserve 85% de la VRAM (15% de marge) |
| `--max-model-len` | 131072 | Contexte maximum supporte |

### KV cache FP8 : compromis vitesse/capacite

| Config KV cache | Capacite tokens | Debit decode | Cas d'usage |
|-----------------|----------------|-------------|-------------|
| fp8 | 335K tokens | 86 tok/s | Multi-utilisateurs, long contexte |
| auto (fp16) | 206K tokens | 96-109 tok/s | Mono-utilisateur, max vitesse |

Le KV cache FP8 permet de stocker **63% de tokens en plus** avec une perte
de vitesse de seulement ~12%. C'est le choix par defaut pour les deployements
multi-utilisateurs.

> **Attention** : Pour les modeles MoE, les kernels Marlin MoE ont une allocation
> memoire variable qui peut causer des erreurs CUDA OOM au runtime
> (bug vLLM #27951). Prevoir une marge de 15% minimum.

In [7]:
# Test des modeles quantifies deployes localement
# Utilise les endpoints configures dans .env

from openai import OpenAI

endpoints = []

# Endpoint mini (ZwZ-8B AWQ)
mini_key = os.getenv("OPENAI_API_KEY_2")
mini_url = os.getenv("OPENAI_BASE_URL_2")
mini_model = os.getenv("OPENAI_CHAT_MODEL_ID_2")
if mini_key and mini_url:
    endpoints.append(("mini (ZwZ-8B AWQ)", mini_url, mini_key, mini_model))

# Endpoint medium (Qwen3.5 GPTQ)
med_key = os.getenv("OPENAI_API_KEY_3")
med_url = os.getenv("OPENAI_BASE_URL_3")
med_model = os.getenv("OPENAI_CHAT_MODEL_ID_3")
if med_key and med_url:
    endpoints.append(("medium (Qwen3.5 GPTQ)", med_url, med_key, med_model))

for name, base_url, api_key, model in endpoints:
    print(f"\n=== Test {name} ===")
    try:
        client = OpenAI(base_url=base_url, api_key=api_key)

        # Lister les modeles disponibles
        models = client.models.list()
        print(f"Modeles disponibles: {[m.id for m in models.data]}")

        # Test de chat
        t0 = time.time()
        resp = client.chat.completions.create(
            model=model or models.data[0].id,
            messages=[{
                "role": "user",
                "content": "Explique la quantization des LLMs en 2 phrases."
            }],
            max_tokens=100,
        )
        elapsed = time.time() - t0
        tokens = resp.usage.completion_tokens

        print(f"Reponse: {resp.choices[0].message.content}")
        print(f"Tokens: {tokens}, Temps: {elapsed:.2f}s, Vitesse: {tokens/elapsed:.1f} tok/s")

    except Exception as e:
        print(f"Erreur: {e}")

if not endpoints:
    pass
# Chargement robuste de la configuration .env
from dotenv import load_dotenv
import os
# Recherche du .env dans tous les parents (pour Papermill qui change le cwd)
current_path = Path.cwd()
env_loaded = False
while current_path.name != "GenAI" and len(current_path.parts) > 1:
    env_path = current_path / ".env"
    if env_path.exists():
        load_dotenv(env_path)
        print(f".env charge depuis: {env_path}")
        env_loaded = True
        break
    current_path = current_path.parent
if not env_loaded:
    print("WARNING: .env non trouve, utilisation variables environnement")
    print("Configurez OPENAI_API_KEY_2/3, OPENAI_BASE_URL_2/3, OPENAI_CHAT_MODEL_ID_2/3")
    print("Voir le fichier .env.example pour les details de configuration")


=== Test mini (ZwZ-8B AWQ) ===


Modeles disponibles: ['zwz-8b']


Reponse: La quantization des LLMs consiste à réduire la précision des poids du modèle (par exemple de float32 à int8) pour diminuer sa taille et accélérer son inference. Cela permet d’exécuter les modèles sur des ressources plus limitées sans sacrifier fortement la qualité de la prédiction.
Tokens: 75, Temps: 0.80s, Vitesse: 94.3 tok/s

=== Test medium (Qwen3.5 GPTQ) ===


Modeles disponibles: ['qwen3.5-35b-a3b']


Reponse: Thinking Process:

1.  **Analyze the Request:**
    *   Topic: Quantization of Large Language Models (LLMs).
    *   Language: French.
    *   Constraint: Exactly 2 sentences (phrases).

2.  **Define LLM Quantization:**
    *   What is it? Reducing the precision of model weights (e.g., from 32-bit floating point to 8-bit, 4-bit, etc.).

Tokens: 100, Temps: 1.94s, Vitesse: 51.5 tok/s


In [ ]:
# === WORKAROUND: Pydantic 2.x by_alias bug ===
# Ce patch corrige le problème "TypeError: argument 'by_alias': 'NoneType' object cannot be converted to 'PyBool'"
# qui survient avec Pydantic 2.x utilisé par l'API OpenAI

import pydantic
import pydantic.functional_validators

# Sauvegarder la fonction originale
_original_model_dump = pydantic.BaseModel.model_dump

def _patched_model_dump(self, **kwargs):
    """Patch pour forcer by_alias à False si None"""
    if 'by_alias' in kwargs and kwargs['by_alias'] is None:
        kwargs['by_alias'] = False
    return _original_model_dump(self, **kwargs)

# Appliquer le patch
pydantic.BaseModel.model_dump = _patched_model_dump

print('✓ Pydantic by_alias workaround applied')


### Interpretation : deploiement vLLM

**Scenario 1** : Endpoints configures

Si les endpoints locaux sont configures dans `.env`, la cellule affiche :
- La liste des modeles servis par chaque instance vLLM
- La reponse generee et la vitesse en tokens/seconde
- Le temps de premiere reponse (TTFT)

**Scenario 2** : Pas d'endpoints

Si aucun endpoint n'est configure, c'est normal :
les endpoints vLLM locaux ne sont disponibles que sur la machine de production
(3x RTX 4090). En cours, l'enseignant peut les fournir via `.env`.

**Points cles** :
1. L'API vLLM est **100% compatible OpenAI** : meme client, meme code
2. Le changement cloud -> local se fait en changeant uniquement `base_url` et `api_key`
3. La vitesse depend du modele, de la precision, et du nombre d'utilisateurs simultanes

---

## Section 6 : Validation qualite -- Benchmarks reels

La question fondamentale est : **la quantization 4-bit degrade-t-elle la qualite ?**

Voici les resultats de benchmarks reels executes sur notre stack de production
(mars 2026, vLLM avec modeles AWQ/GPTQ 4-bit).

### Point pedagogique cle

ZwZ-8B (8B parametres, specialise vision) bat Qwen3.5 (35B MoE) sur MMStar
(63% vs 53%). Cela montre que la **specialisation par fine-tuning** peut compenser
la taille du modele. Mais sur MME (benchmark plus large), le modele plus gros
reprend l'avantage.

In [8]:
# Resultats de benchmarks reels (workspace vLLM, mars 2026)
# Ces resultats sont sur des modeles AWQ/GPTQ 4-bit en production

print("=" * 70)
print("BENCHMARKS QUALITE - Modeles quantifies 4-bit (production)")
print("=" * 70)

# Donnees issues des benchmarks de production
benchmarks = {
    "GSM8K (math CoT 0-shot)": {"Qwen3.5 GPTQ": "88.0%", "ZwZ-8B AWQ": "N/A"},
    "IFEval (instruction following)": {"Qwen3.5 GPTQ": "88.5%", "ZwZ-8B AWQ": "N/A"},
    "MME (vision perception+cognition)": {"Qwen3.5 GPTQ": "1294.7 (91%)", "ZwZ-8B AWQ": "1248.1 (89%)"},
    "MMStar (vision multi-choice)": {"Qwen3.5 GPTQ": "53.2%", "ZwZ-8B AWQ": "63.0% (!)"},
}

print(f"\n{'Benchmark':<35} {'Qwen3.5 (35B MoE)':>18} {'ZwZ-8B (8B)':>15}")
print("-" * 70)
for bench, scores in benchmarks.items():
    print(f"{bench:<35} {scores['Qwen3.5 GPTQ']:>18} {scores['ZwZ-8B AWQ']:>15}")

print("\n" + "=" * 70)
print("BENCHMARKS PERFORMANCE - Vitesse d'inference")
print("=" * 70)

perf = [
    ("Decode single-user", "86.2 tok/s", "135 tok/s"),
    ("Concurrent 5 users", "269.6 tok/s", "392 tok/s"),
    ("Context max", "262K tokens", "131K tokens"),
    ("KV cache (FP8)", "335K tokens", "N/A"),
]

print(f"\n{'Metrique':<25} {'Qwen3.5 (TP=2)':>18} {'ZwZ-8B (1 GPU)':>18}")
print("-" * 65)
for metric, q35, zwz in perf:
    print(f"{metric:<25} {q35:>18} {zwz:>18}")

print("\n--- Point pedagogique cle ---")
print("ZwZ-8B (8B params) bat Qwen3.5 (35B MoE) sur MMStar (63% vs 53%)!")
print("La specialisation par fine-tuning peut compenser la taille du modele.")
print("Mais sur MME (benchmark plus large), le modele plus gros reprend l'avantage.")

BENCHMARKS QUALITE - Modeles quantifies 4-bit (production)

Benchmark                            Qwen3.5 (35B MoE)     ZwZ-8B (8B)
----------------------------------------------------------------------
GSM8K (math CoT 0-shot)                          88.0%             N/A
IFEval (instruction following)                   88.5%             N/A
MME (vision perception+cognition)         1294.7 (91%)    1248.1 (89%)
MMStar (vision multi-choice)                     53.2%       63.0% (!)

BENCHMARKS PERFORMANCE - Vitesse d'inference

Metrique                      Qwen3.5 (TP=2)     ZwZ-8B (1 GPU)
-----------------------------------------------------------------
Decode single-user                86.2 tok/s          135 tok/s
Concurrent 5 users               269.6 tok/s          392 tok/s
Context max                      262K tokens        131K tokens
KV cache (FP8)                   335K tokens                N/A

--- Point pedagogique cle ---
ZwZ-8B (8B params) bat Qwen3.5 (35B MoE) sur MMSta

### Interpretation : benchmarks

**Qualite (AWQ/GPTQ 4-bit)** :

| Benchmark | Qwen3.5 GPTQ | ZwZ-8B AWQ | Commentaire |
|-----------|-------------|-----------|-------------|
| GSM8K (math) | 88.0% | N/A | Excellent pour un 4-bit |
| IFEval (instructions) | 88.5% | N/A | Bonne qualite de suivi d'instructions |
| MME (vision) | 1294.7 | 1248.1 | Qwen3.5 > ZwZ-8B (taille superieure) |
| MMStar (vision multi-choice) | 53.2% | 63.0% | ZwZ-8B gagne (specialisation) |

**Performance (vitesse)** :

| Metrique | Qwen3.5 (TP=2) | ZwZ-8B (1 GPU) |
|----------|----------------|----------------|
| Decode single-user | 86.2 tok/s | 135 tok/s |
| Concurrent 5 users | 269.6 tok/s | 392 tok/s |
| TTFT 30K tokens | 0.95s | N/A |
| Tool calling | 893ms | N/A |

**Points cles** :
1. La quantization 4-bit preserve remarquablement bien la qualite (88% sur GSM8K)
2. ZwZ-8B (specialise vision) bat le modele 4x plus gros sur MMStar
3. Le modele plus petit (8B, 1 GPU) est 1.5x plus rapide que le MoE (35B, 2 GPUs)
4. Le choix du modele depend du cas d'usage : specialise vs generaliste

> **Conclusion pratique** : La quantization 4-bit est un **no-brainer** pour la production.
> La perte de qualite est marginale, les gains en memoire et vitesse sont majeurs.

---

## Section 7 : AWQ vs GPTQ -- Comparaison sur le meme modele

Pour comprendre les differences pratiques entre AWQ et GPTQ, comparons deux
quantizations 4-bit du meme modele de base : **Qwen3.5-35B-A3B**.

- **AWQ** : Version communautaire (cyankiwi), format compressed-tensors
- **GPTQ-Int4** : Version officielle Qwen, format GPTQ standard

### Enjeu : Multi-Token Prediction (MTP)

Le MTP est une technique de decodage speculatif ou un "draft model" predit
plusieurs tokens a l'avance. Si les predictions sont acceptees, on gagne
en vitesse. Le taux d'acceptation depend de la fidelite des logits.

In [9]:
# Comparaison AWQ vs GPTQ sur le meme modele de base
# Qwen3.5-35B-A3B : deux quantizations 4-bit differentes

print("=" * 70)
print("AWQ vs GPTQ - Meme modele, methodes differentes")
print("=" * 70)

comparison = [
    ("Modele source", "Qwen3.5-35B-A3B", "Qwen3.5-35B-A3B"),
    ("Quantization", "AWQ (cyankiwi)", "GPTQ-Int4 (officiel Qwen)"),
    ("Format", "compressed-tensors", "GPTQ standard"),
    ("Kernels vLLM", "Marlin AWQ (fused dequant)", "Marlin GPTQ (WNA16)"),
    ("MTP (speculative)", "0% acceptance (!)", "Potentiellement fonctionnel"),
    ("Auto-detection vLLM", "Oui (pas de flag)", "Flag --quantization moe_wna16"),
    ("Createur", "Communaute", "Equipe Qwen officielle"),
]

print(f"\n{'Critere':<25} {'AWQ':>25} {'GPTQ':>30}")
print("-" * 82)
for critere, awq, gptq in comparison:
    print(f"{critere:<25} {awq:>25} {gptq:>30}")

print("\n--- Pourquoi MTP ne marche pas avec AWQ ? ---")
print("Multi-Token Prediction utilise un draft model pour predire les tokens suivants.")
print("La precision 4-bit AWQ introduit trop de bruit dans les logits,")
print("rendant les predictions du draft quasi-aleatoires (0% acceptance).")
print("Le GPTQ standard semble mieux preserver les distributions de logits.")

AWQ vs GPTQ - Meme modele, methodes differentes

Critere                                         AWQ                           GPTQ
----------------------------------------------------------------------------------
Modele source                       Qwen3.5-35B-A3B                Qwen3.5-35B-A3B
Quantization                         AWQ (cyankiwi)      GPTQ-Int4 (officiel Qwen)
Format                           compressed-tensors                  GPTQ standard
Kernels vLLM              Marlin AWQ (fused dequant)            Marlin GPTQ (WNA16)
MTP (speculative)                 0% acceptance (!)    Potentiellement fonctionnel
Auto-detection vLLM               Oui (pas de flag)  Flag --quantization moe_wna16
Createur                                 Communaute         Equipe Qwen officielle

--- Pourquoi MTP ne marche pas avec AWQ ? ---
Multi-Token Prediction utilise un draft model pour predire les tokens suivants.
La precision 4-bit AWQ introduit trop de bruit dans les logits,
rendant les 

### Interpretation : AWQ vs GPTQ

**Resume de la comparaison** :

| Aspect | AWQ | GPTQ | Gagnant |
|--------|-----|------|--------|
| Facilite de deploiement | Auto-detecte | Flag necessaire | AWQ |
| MTP (decodage speculatif) | 0% acceptance | Fonctionnel | GPTQ |
| Ecosysteme | Communautaire | Officiel Qwen | GPTQ |
| Kernels vLLM | Marlin AWQ | Marlin GPTQ | Equivalent |

**Points cles** :
1. Les deux methodes produisent des modeles de qualite comparable en inference standard
2. La difference apparait sur les techniques avancees comme le MTP
3. Pour un deploiement simple sans MTP, les deux conviennent
4. Pour exploiter le MTP (gain de vitesse potentiel de 30-50%), privilegier GPTQ

> **Recommandation pratique** : Pour un nouveau deploiement, verifier d'abord
> si une version GPTQ officielle existe sur HuggingFace. Sinon, utiliser
> llmcompressor pour creer sa propre quantization GPTQ.

---

# Conclusion

## Trois lecons a retenir

1. **La quantization accelere l'inference** : Le bottleneck des LLMs est la bande passante
   memoire, pas le compute. Reduire les poids de 16 bits a 4 bits divise par 4 les donnees
   a transferer, ce qui accelere la generation de tokens.

2. **Les modeles multimodaux necessitent des exclusions selectrices** : Ne jamais quantifier
   l'encodeur vision (ViT/SigLIP) car les operations softmax et LayerNorm sont extremement
   sensibles a la precision. Le cout en memoire est negligeable (~0.7 GB pour un modele 8B VL).

3. **Le format de sortie determine le runtime** : compressed-tensors -> Marlin (vLLM),
   GPTQ -> GPTQMarlin (vLLM), GGUF -> llama.cpp/Ollama. Choisissez le format
   selon votre runtime cible.

## Tableau recapitulatif

| Methode | Cas d'usage | Difficulte | Runtime |
|---------|-------------|-----------|--------|
| BitsAndBytes | Prototypage, QLoRA | Tres simple | Transformers |
| AutoAWQ | Production, communautaire | Simple | vLLM, Transformers |
| GPTQ (llmcompressor) | Production (recommande) | Moyen | vLLM (natif) |
| GGUF | CPU, edge, Ollama | Simple | llama.cpp |

## Ressources

- [Documentation vLLM - Quantization](https://docs.vllm.ai/en/latest/quantization/)
- [llmcompressor GitHub](https://github.com/vllm-project/llm-compressor)
- [Blog Neural Magic - Quantization](https://neuralmagic.com/blog/)
- [HuggingFace - Quantization Guide](https://huggingface.co/docs/transformers/quantization)
- [Paper GPTQ](https://arxiv.org/abs/2210.17323)
- [Paper AWQ](https://arxiv.org/abs/2306.00978)

---

## Exercices pratiques

### Exercice 1 : Calcul d'empreinte memoire (facile)

Utilisez la fonction `estimate_model_memory()` pour calculer l'empreinte
des modeles suivants en BF16, INT8 et W4A16 :
- Mistral 7B
- Mixtral 8x7B (56B total, 12B actifs)
- Llama 3.1 405B

Combien de RTX 4090 (24 GB) faut-il pour chaque configuration ?

### Exercice 2 : Quantification avec llmcompressor (intermediaire)

Adaptez le script de la section 3 pour quantifier `Qwen/Qwen3.5-0.8B` :
1. Creez un environnement conda dedie
2. Installez les dependances
3. Executez la quantization
4. Mesurez la taille du repertoire avant/apres
5. Verifiez que le modele se charge sans erreur

### Exercice 3 : Deploiement vLLM (intermediaire)

Deployez votre modele quantifie avec vLLM :
1. Lancez `vllm serve` sur le modele quantifie
2. Testez avec un client OpenAI
3. Mesurez les tokens/seconde
4. Comparez avec le modele BF16 original

### Exercice 4 : Perplexite avant/apres (avance)

Comparez la perplexite du modele original et quantifie sur WikiText-2 :
```python
from datasets import load_dataset
ds = load_dataset("wikitext", "wikitext-2-raw-v1", split="test")
# Calculer la perplexite pour les deux versions
```

### Exercice 5 : Exclusion vision (avance)

Explorez les couches d'un modele VL :
1. Chargez `Qwen/Qwen3-VL-2B` avec `AutoModel.from_pretrained()`
2. Listez toutes les couches avec `named_modules()`
3. Identifiez les couches `visual` et `merger`
4. Calculez le pourcentage de parametres qu'elles representent